# Zyro Dynamics HR Help Desk — RAG Challenge
### NxtWave Masterclass | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Deploy a Streamlit chatbot
- Generate your `submission.csv`

## Submission Requirements

1. `submission.csv` — upload on Kaggle
2. Streamlit App URL
3. LangSmith Trace URL

---

> Follow the notebook cells sequentially and complete the sections marked for implementation.

## Cell 1 — Install Dependencies

> ⚠️ Run this cell first before anything else.

This cell installs all required libraries for:
- document loading
- embeddings
- vector database
- RAG pipeline
- Streamlit deployment
- LangSmith tracing

> After installation completes, restart the kernel/runtime and run all cells from the top.

In [ ]:
print("Installing required packages...\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    streamlit \
    langsmith \
    python-dotenv \
    tiktoken

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

## Cell 2 — Configuration

This is the main configuration cell for the notebook.

Here you can:
- choose your LLM provider
- select the model you want to use
- update related settings if needed

All remaining cells will automatically use this configuration.

In [ ]:
LLM_PROVIDER = "groq"  # "groq" | "gemini" | "openai"
LLM_MODEL = "llama-3.3-70b-versatile"  # change model here if needed

CORPUS_PATH = "/kaggle/input/zyro-dynamics-hr-corpus/"

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")

## Cell 3 — Imports

This cell imports all required libraries for:
- document loading
- text chunking
- embeddings
- vector search
- prompt handling
- LangSmith tracing

> Run this cell without modifying anything.

In [ ]:
import os, json, time, csv
from cryptography.fernet import Fernet
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

print("Imports loaded successfully.")

## Cell 4 — API Keys + LangSmith Setup

This cell loads:
- your LLM API key
- LangSmith API key
- environment configuration

LangSmith tracing is enabled automatically for monitoring and debugging your RAG pipeline.

> Add the required API keys before running this cell.
> This section is pre-filled — no modifications needed.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    if LLM_PROVIDER == "groq":
        os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    elif LLM_PROVIDER == "gemini":
        os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    elif LLM_PROVIDER == "openai":
        os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

    os.environ["LANGCHAIN_API_KEY"]    = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"
    print("Running on Kaggle — secrets loaded!")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "#your project Name"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully.")

## Cell 5 — Load Documents

### Your Task

Load all policy documents from the provided corpus directory using a PDF loader.

Store the loaded documents and print the total number loaded.

In [ ]:
# Initialize document loader
loader = PyPDFDirectoryLoader(CORPUS_PATH)

# Load documents
documents = loader.load()

print(f"Loaded {len(documents)} documents")

## Cell 6 — Chunk Documents

### Your Task

Split the loaded documents into smaller chunks using `RecursiveCharacterTextSplitter`.

Store the generated chunks and print the total number created.

In [ ]:
# Initialize text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Create document chunks
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

## Cell 7 — Embeddings

### Your Task

Initialize an embedding model to convert document chunks into vector representations.

Use `HuggingFaceEmbeddings` for the implementation below.

In [ ]:
# Choose and initialize an embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model initialized.")

## Cell 8 — Vector Store + Retriever

### Your Task

Build a FAISS vector store using the generated chunks and embeddings.

Then create a retriever for retrieving relevant document chunks during question answering.

In [ ]:
# Build vector store from document chunks and embeddings
vectorstore = FAISS.from_documents(chunks, embeddings)

# Create retriever using MMR for diverse, relevant results
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

print("Vector store initialized.")

## Cell 9 — LLM Initialization

The language model is initialized automatically using the configuration from Cell 2.

You may optionally adjust parameters such as:
- `temperature`
- `max_tokens`

based on your preferred response style and performance.

> This cell is pre-filled — modify only if needed.

In [ ]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_output_tokens=512
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

else:
    raise ValueError("Unsupported LLM provider.")

print("LLM initialized.")

## Cell 10 — Build the RAG Chain

### Your Task

Implement the RAG pipeline using LCEL and enable LangSmith tracing.

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful HR assistant for Zyro Dynamics Pvt. Ltd.\n"
    "Answer the employee's question using ONLY the information provided in the context below.\n\n"
    "If the context does not contain sufficient information, respond with:\n"
    "\"I don't have enough information in the HR policy documents to answer that question.\"\n\n"
    "Be concise, accurate, and professional. Reference the relevant policy where appropriate.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)


def format_docs(docs):
    """Format retrieved document chunks into a context string with source citations."""
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "Unknown").split("/")[-1]
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        parts.append(f"[Source: {source}, Page: {page}]\n{doc.page_content}")
    return "\n\n".join(parts)


@traceable(name="rag_chain")
def rag_chain(question: str) -> dict:
    """Retrieve relevant HR policy chunks and generate a grounded answer."""
    docs = retriever.invoke(question)
    context = format_docs(docs)

    chain = (
        RunnablePassthrough.assign(context=lambda x: x["context"])
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke({"question": question, "context": context})
    sources = list({
        doc.metadata.get("source", "Unknown").split("/")[-1]
        for doc in docs
        if doc.metadata.get("source")
    })

    return {"answer": answer, "sources": sources, "context_docs": docs}


print("RAG pipeline initialized.")

## Cell 11 — Guardrails

### Your Task

Implement handling for out-of-scope questions before routing queries through the RAG pipeline.

In [ ]:
OOS_PROMPT = ChatPromptTemplate.from_template(
    "You are a classifier that determines whether an employee's question can be\n"
    "answered using Zyro Dynamics' internal HR policy documents.\n\n"
    "The HR documents cover ONLY these topics:\n"
    "1. Company profile and culture\n"
    "2. Employee handbook and general workplace guidelines\n"
    "3. Leave policy (EL, CL, SL, Maternity, Paternity, Bereavement, Comp-off)\n"
    "4. Work from home and hybrid/remote work policy\n"
    "5. Code of conduct and ethics\n"
    "6. Performance reviews, PIP, ratings, and promotions\n"
    "7. Compensation, salary structure, benefits, and insurance\n"
    "8. IT and data security policy\n"
    "9. Prevention of sexual harassment (POSH) and ICC\n"
    "10. Onboarding, probation, separation, and notice periods\n"
    "11. Travel and expense reimbursements\n\n"
    "Classify as OUT_OF_SCOPE if the question:\n"
    "- Is about a topic NOT listed above (e.g. recruitment/hiring process for\n"
    "  external job applicants, company financial performance or revenue,\n"
    "  detailed product feature comparisons, other companies' policies,\n"
    "  personal income tax advice)\n"
    "- Asks for personalised individual data not in policy documents\n"
    "  (e.g. 'what is my exact salary', 'how many stock options will I get')\n"
    "- Is general knowledge or unrelated to Zyro Dynamics HR\n\n"
    "Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE\n\n"
    "Question: {question}\n\n"
    "Classification:"
)

REFUSAL_MESSAGE = (
    "I'm sorry, I can only answer HR-related questions based on "
    "Zyro Dynamics' internal policy documents. "
    "Your question appears to be outside the scope of what I can help with. "
    "Please reach out to hr@zyro.com or speak with your HR Business Partner "
    "for further assistance."
)


@traceable(name="ask_bot")
def ask_bot(question: str) -> dict:
    """Route question through guardrail classifier before the RAG pipeline."""
    classifier_chain = OOS_PROMPT | llm | StrOutputParser()
    verdict = classifier_chain.invoke({"question": question}).strip().upper()

    if "OUT_OF_SCOPE" in verdict:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": [],
            "in_scope": False,
        }

    result = rag_chain(question)
    result["in_scope"] = True
    return result


print("Guardrails initialized.")

## Cell 12 — Test the Bot

### Your Task

Test your RAG pipeline using a few sample questions of your choice.

In [ ]:
test_questions = [
    "How many days of Casual Leave am I entitled to per year, and can unused CL be carried forward?",
    "What are the eligibility criteria for working from home in a hybrid arrangement?",
    "Can you help me file my income tax returns for this financial year?",
]

for i, q in enumerate(test_questions, 1):
    print(f"Q{i}: {q}")
    result = ask_bot(q)
    print(f"Answer: {result['answer']}")
    if result.get("sources"):
        print(f"Sources: {', '.join(result['sources'])}")
    print(f"In Scope: {result.get('in_scope', 'N/A')}")
    print("-" * 60)

## Cell 13 — LangSmith Trace URL

Generate and copy your shareable LangSmith trace URL for submission.

> This cell is pre-filled — just run it and follow the instructions.

In [ ]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")

## Cell 14 — Streamlit App

### Your Task

Build and deploy a Streamlit chatbot application for your RAG pipeline.

Save your implementation as `app.py`.

In [ ]:
app_code = '''
import os
import streamlit as st
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

st.set_page_config(
    page_title="Zyro Dynamics HR Help Desk",
    page_icon="🏢",
    layout="centered",
)

CORPUS_PATH = os.getenv("CORPUS_PATH", "./data/")

@st.cache_resource(show_spinner="Loading HR policy documents...")
def build_pipeline():
    loader = PyPDFDirectoryLoader(CORPUS_PATH)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=[chr(10)+chr(10), chr(10), ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5},
    )
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1, max_tokens=512)
    return retriever, llm

RAG_TEMPLATE = """You are a helpful HR assistant for Zyro Dynamics Pvt. Ltd.
Answer the employee question using ONLY the information in the context below.

If the context does not contain enough information, say:
"I don't have enough information in the HR policy documents to answer that."

Be concise, accurate, and professional.

Context:
{context}

Question: {question}

Answer:"""

OOS_TEMPLATE = """You are a classifier. Determine if the employee question can be answered
using Zyro Dynamics internal HR policy documents.

In-scope topics: leave policy, WFH/hybrid policy, compensation and benefits,
performance reviews, code of conduct, IT security, POSH, onboarding and
separation, travel expenses, company overview and culture.

Classify as OUT_OF_SCOPE if the question is about: recruitment/hiring for
job applicants, company financial performance, detailed product comparisons,
other companies policies, personal tax filing, or personalized individual
data not in policy documents.

Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE

Question: {question}

Classification:"""

REFUSAL = (
    "I can only answer HR-related questions based on "
    "Zyro Dynamics internal policy documents. "
    "This question is outside my scope. "
    "Please contact hr@zyro.com for further assistance."
)

def format_docs(docs):
    nl = chr(10)
    parts = []
    for doc in docs:
        src = doc.metadata.get("source", "Unknown").split("/")[-1]
        pg = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        header = "[Source: " + src + ", Page: " + str(pg) + "]"
        parts.append(header + nl + doc.page_content)
    return (nl + nl).join(parts)

def ask_bot(question, retriever, llm):
    oos_prompt = ChatPromptTemplate.from_template(OOS_TEMPLATE)
    oos_chain = oos_prompt | llm | StrOutputParser()
    verdict = oos_chain.invoke({"question": question}).strip().upper()
    if "OUT_OF_SCOPE" in verdict:
        return {"answer": REFUSAL, "sources": [], "in_scope": False}
    docs = retriever.invoke(question)
    context = format_docs(docs)
    rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)
    chain = (
        RunnablePassthrough.assign(context=lambda x: x["context"])
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    answer = chain.invoke({"question": question, "context": context})
    sources = list({
        doc.metadata.get("source", "").split("/")[-1]
        for doc in docs
        if doc.metadata.get("source")
    })
    return {"answer": answer, "sources": sources, "in_scope": True}

# ── Sidebar ──────────────────────────────────────────────────────────────────
with st.sidebar:
    st.title("HR Help Desk")
    st.markdown("Ask questions about **Zyro Dynamics HR policies**")
    st.divider()
    for topic in [
        "Leave & attendance",
        "Work from home rules",
        "Compensation & benefits",
        "Performance reviews",
        "Code of conduct",
        "IT & security policy",
        "POSH guidelines",
        "Onboarding & separation",
        "Travel & expense",
    ]:
        st.markdown("- " + topic)
    st.divider()
    if st.button("Clear Chat"):
        st.session_state.messages = []
        st.rerun()

# ── Main ─────────────────────────────────────────────────────────────────────
st.title("Zyro Dynamics HR Help Desk")
st.caption("Powered by RAG - Ask questions about company HR policies")

try:
    groq_key = os.getenv("GROQ_API_KEY") or st.secrets.get("GROQ_API_KEY", "")
    if groq_key:
        os.environ["GROQ_API_KEY"] = groq_key
    else:
        st.error("GROQ_API_KEY not configured. Add it to Streamlit secrets.")
        st.stop()
except Exception:
    if not os.getenv("GROQ_API_KEY"):
        st.error("GROQ_API_KEY not configured. Add it to Streamlit secrets.")
        st.stop()

retriever, llm = build_pipeline()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])
        if msg.get("sources"):
            with st.expander("Source Documents"):
                for s in msg["sources"]:
                    st.write("- " + s)

if prompt := st.chat_input("Ask an HR question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Searching HR policies..."):
            result = ask_bot(prompt, retriever, llm)
        st.write(result["answer"])
        if result.get("sources"):
            with st.expander("Source Documents"):
                for s in result["sources"]:
                    st.write("- " + s)
    st.session_state.messages.append({
        "role": "assistant",
        "content": result["answer"],
        "sources": result.get("sources", []),
    })
'''

with open("app.py", "w") as f:
    f.write(app_code.strip())

print("app.py created.")

## Cell 15 — Evaluation

Evaluation inputs are loaded automatically at runtime.

> Do not modify this cell.

In [ ]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ=="),
]

eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

## Cell 16 — Generate `submission.csv`

Generate your final `submission.csv` file for submission.

> Do not modify this cell.

In [ ]:
import re

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = input("Streamlit App URL: ").strip()
langsmith_link = input("LangSmith Trace URL: ").strip()

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

## Cell 17 — Final Checklist

Verify your submission files and links before submitting on Kaggle.

> This cell is pre-filled — just run it.

In [ ]:
import re, csv, os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")